# General Preprocessing Pipeline
This notebook performs the shared preprocessing steps for all models to ensure fair comparison.
1. Drop 'activity' column
2. Label Encode 'label' column
3. Stratified Split (80% Train, 10% Validation, 10% Test)
4. Zero-variance feature removal
5. Highly-correlated feature removal (>0.95)

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
DATASET_PATH = "../dataset/bccc-cpacket-cloud-ddos-2024-merged.parquet"
print(f"Loading dataset from {DATASET_PATH}...")
df = pd.read_parquet(DATASET_PATH)
print(f"Original shape: {df.shape}")

Loading dataset from ../dataset/bccc-cpacket-cloud-ddos-2024-merged.parquet...
Original shape: (540494, 319)


In [3]:
if 'activity' in df.columns:
    df = df.drop(columns=['activity'])
    print("Dropped 'activity' column.")

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

print("Label mapping:")
for cls in le.classes_:
    print(f"  {cls} -> {int(le.transform([cls])[0])}")

Dropped 'activity' column.
Label mapping:
  Attack -> 0
  Benign -> 1
  Suspicious -> 2


In [4]:
X = df.drop(columns=['label'])
y = df['label']

# First split: 80% Train, 20% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Second split: 50% of Temp -> 10% Val, 10% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train shape: {X_train.shape}")
print(f"Val shape:   {X_val.shape}")
print(f"Test shape:  {X_test.shape}")

Train shape: (432395, 317)
Val shape:   (54049, 317)
Test shape:  (54050, 317)


In [5]:
zero_var_cols = [col for col in X_train.columns if X_train[col].nunique(dropna=False) <= 1]

if zero_var_cols:
    X_train = X_train.drop(columns=zero_var_cols)
    X_val = X_val.drop(columns=zero_var_cols)
    X_test = X_test.drop(columns=zero_var_cols)

print(f"Dropped {len(zero_var_cols)} zero-variance columns.")

Dropped 3 zero-variance columns.


In [6]:
sample_n = min(50_000, len(X_train))
corr_sample = X_train.sample(n=sample_n, random_state=42)
corr_matrix = corr_sample.corr().abs()

upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]

if high_corr_cols:
    X_train = X_train.drop(columns=high_corr_cols)
    X_val = X_val.drop(columns=high_corr_cols)
    X_test = X_test.drop(columns=high_corr_cols)

print(f"Dropped {len(high_corr_cols)} highly-correlated columns (>0.95).")
print(f"Final Feature Space: {X_train.shape[1]} columns")

Dropped 152 highly-correlated columns (>0.95).
Final Feature Space: 162 columns


In [7]:
PROCESSED_DIR = "../dataset/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

X_train.to_parquet(f"{PROCESSED_DIR}/X_train.parquet")
X_val.to_parquet(f"{PROCESSED_DIR}/X_val.parquet")
X_test.to_parquet(f"{PROCESSED_DIR}/X_test.parquet")

pd.DataFrame(y_train).to_parquet(f"{PROCESSED_DIR}/y_train.parquet")
pd.DataFrame(y_val).to_parquet(f"{PROCESSED_DIR}/y_val.parquet")
pd.DataFrame(y_test).to_parquet(f"{PROCESSED_DIR}/y_test.parquet")

preprocessing_info = {
    "zero_var_cols": zero_var_cols,
    "high_corr_cols": high_corr_cols,
    "feature_columns": list(X_train.columns)
}
joblib.dump(preprocessing_info, f"{PROCESSED_DIR}/preprocessing_info.joblib")
joblib.dump(le, f"{PROCESSED_DIR}/label_encoder.joblib")

print(f"Successfully saved processed datasets and preprocessing info to '{PROCESSED_DIR}'")

Successfully saved processed datasets and preprocessing info to '../dataset/processed'
